# SatQuery AI — Division 3: Large-Scale Real LEVIR-CD Training & Benchmark Pipeline

**Project**: SatQuery AI (Agentic Vision-Language System for Multi-Sensor Remote Sensing)  
**Division**: Division 3 — Bi-Temporal Change Intelligence  
**Model Architecture**: TinyCD (Siamese U-Net + MAMB Space-Time Attention Block)  
**Target Dataset**: Full Official LEVIR-CD (445 train / 64 val / 128 test parent scenes)  
**Target Hardware**: NVIDIA Tesla T4 / CUDA 12.x on Google Colab  
**License**: Non-commercial and research purposes only (Andrea Codegoni et al.).

In [ ]:
# CELL 1: GPU Verification & CUDA Telemetry
import torch
import sys

print(f"Python Version:  {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
    print(f"Total VRAM:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: No CUDA GPU detected. Please change Colab Runtime to T4 GPU.")

In [ ]:
# CELL 2: NVIDIA System Management Interface (nvidia-smi)
!nvidia-smi

In [ ]:
# CELL 3: Clone Repository & Configure Workspace
import os
import sys
from pathlib import Path

if not os.path.exists('/content/SatQuery'):
    !git clone https://github.com/Lalith2007/SatQuery.git /content/SatQuery

%cd /content/SatQuery
!git checkout feature/dheeraj-change
!git log -n 1 --oneline

for p in ['/content/SatQuery', '.']:
    abs_p = os.path.abspath(p)
    if abs_p not in sys.path:
        sys.path.insert(0, abs_p)

In [ ]:
# CELL 4: Install Dependencies
!pip install --quiet huggingface_hub pydantic-settings torch torchvision torchaudio tifffile pillow pydantic fastapi pytest pytest-asyncio

In [ ]:
# CELL 5: Automated Download, Unpacking & Structure Resolution for LEVIR-CD
import os
import sys
import zipfile
import tarfile
from pathlib import Path
from huggingface_hub import snapshot_download

# Secure credential retrieval from Colab Secrets (HF_TOKEN) or environment
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.getenv("HF_TOKEN", None)
dataset_root = Path('/content/LEVIR-CD')
dataset_root.mkdir(parents=True, exist_ok=True)

# Check if images already exist
has_train_images = any(dataset_root.rglob("*.png")) or any(dataset_root.rglob("*.jpg"))

if not has_train_images:
    print("Downloading official LEVIR-CD benchmark dataset from HuggingFace into /content/LEVIR-CD...")
    try:
        snapshot_download(
            repo_id="satellite-image-deep-learning/LEVIR-CD",
            repo_type="dataset",
            local_dir=str(dataset_root),
            local_dir_use_symlinks=False,
            token=hf_token,
        )
        print("Download completed!")
    except Exception as e:
        print(f"HuggingFace download notice: {e}")

# Automatically extract any zip/tar files present
archives = list(dataset_root.rglob("*.zip")) + list(dataset_root.rglob("*.tar.gz")) + list(dataset_root.rglob("*.tar"))
for arc in archives:
    print(f"Extracting archive: {arc.name}...")
    try:
        if arc.suffix == ".zip":
            with zipfile.ZipFile(arc, 'r') as zf:
                zf.extractall(arc.parent)
        elif ".tar" in arc.name:
            with tarfile.open(arc, 'r:*') as tf:
                tf.extractall(arc.parent)
    except Exception as ex:
        print(f"Could not extract {arc}: {ex}")

# Auto-discover root containing 'train' split
resolved_root = dataset_root
for cand in [dataset_root] + sorted(list(dataset_root.rglob("*"))):
    if cand.is_dir() and (cand / "train").exists():
        resolved_root = cand
        break

dataset_root = resolved_root
print(f"\nResolved Active LEVIR-CD Dataset Root: {dataset_root}")

if dataset_root.exists():
    subdirs = [d.name for d in dataset_root.iterdir() if d.is_dir()]
    print(f"Subdirectories under dataset root: {subdirs}")

In [ ]:
# CELL 6: Resilient Dynamic Dataset Discovery with Scene-Grouped Split Support
import hashlib
import json
import os
from pathlib import Path
import random
import re
import time
from typing import Any, Dict, List, Optional, Sequence, Tuple
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset

class TemporalChangeDataset(Dataset):
    def __init__(self, samples: List[Dict[str, Any]], image_size: int = 256, is_training: bool = False):
        self.samples = samples
        self.image_size = image_size
        self.is_training = is_training

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        item = self.samples[idx]
        t0_img = Image.open(item["t0_path"]).convert("RGB")
        t1_img = Image.open(item["t1_path"]).convert("RGB")
        if "mask_path" in item and item["mask_path"] and os.path.exists(item["mask_path"]):
            mask_img = Image.open(item["mask_path"]).convert("L")
        else:
            mask_img = Image.new("L", t0_img.size, color=0)

        if t0_img.size != (self.image_size, self.image_size):
            t0_img = t0_img.resize((self.image_size, self.image_size), Image.BILINEAR)
            t1_img = t1_img.resize((self.image_size, self.image_size), Image.BILINEAR)
            mask_img = mask_img.resize((self.image_size, self.image_size), Image.NEAREST)

        t0_arr = np.array(t0_img, dtype=np.float32) / 255.0
        t1_arr = np.array(t1_img, dtype=np.float32) / 255.0
        mask_arr = (np.array(mask_img, dtype=np.float32) > 128.0).astype(np.float32)

        if self.is_training:
            if random.random() > 0.5:
                t0_arr = np.fliplr(t0_arr).copy()
                t1_arr = np.fliplr(t1_arr).copy()
                mask_arr = np.fliplr(mask_arr).copy()
            if random.random() > 0.5:
                t0_arr = np.flipud(t0_arr).copy()
                t1_arr = np.flipud(t1_arr).copy()
                mask_arr = np.flipud(mask_arr).copy()
            rot_k = random.choice([0, 1, 2, 3])
            if rot_k > 0:
                t0_arr = np.rot90(t0_arr, rot_k).copy()
                t1_arr = np.rot90(t1_arr, rot_k).copy()
                mask_arr = np.rot90(mask_arr, rot_k).copy()

        return {
            "t0": torch.from_numpy(t0_arr).permute(2, 0, 1).float(),
            "t1": torch.from_numpy(t1_arr).permute(2, 0, 1).float(),
            "mask": torch.from_numpy(mask_arr).unsqueeze(0).float(),
            "sample_id": item.get("sample_id", f"sample_{idx}"),
            "parent_id": item.get("parent_id", "unknown"),
            "t0_path": item["t0_path"],
            "t1_path": item["t1_path"],
            "mask_path": item.get("mask_path"),
        }

class LEVIRCDDatasetLoader:
    @classmethod
    def extract_parent_id(cls, filename: str, split: str = "train") -> str:
        stem = Path(filename).stem
        m_grid = re.match(r"^([a-zA-Z]+_\d+)_\d+_\d+$", stem)
        if m_grid:
            return m_grid.group(1)
        m_patch = re.match(r"^(.+?)(?:_p\d+|_patch\d+|_sub\d+)?$", stem)
        if m_patch and m_patch.group(1) not in ["", "train", "val", "test", "A", "B", "label", "image"]:
            return m_patch.group(1)
        return f"{split}_{stem}"

    @classmethod
    def discover_split_samples(cls, root_path, split: str = "train", mode: str = "full", dev_limit: Optional[int] = None):
        root = Path(root_path)
        samples = []
        
        split_dirs = [root / split, root / split.lower(), root / split.upper()]
        for d in root.rglob(split):
            if d.is_dir() and d not in split_dirs:
                split_dirs.append(d)
        
        target_split_dir = None
        for sd in split_dirs:
            if sd.exists() and sd.is_dir():
                target_split_dir = sd
                break
        
        if target_split_dir is None:
            return []

        a_dirs = [target_split_dir / "A", target_split_dir / "a", target_split_dir / "time1", target_split_dir / "T1"]
        b_dirs = [target_split_dir / "B", target_split_dir / "b", target_split_dir / "time2", target_split_dir / "T2"]
        l_dirs = [target_split_dir / "label", target_split_dir / "label_256", target_split_dir / "labels", target_split_dir / "mask", target_split_dir / "masks"]
        
        for d in target_split_dir.rglob("*"):
            if d.is_dir():
                if d.name.lower() in ["a", "time1", "t1"] and d not in a_dirs:
                    a_dirs.append(d)
                elif d.name.lower() in ["b", "time2", "t2"] and d not in b_dirs:
                    b_dirs.append(d)
                elif d.name.lower() in ["label", "label_256", "labels", "mask"] and d not in l_dirs:
                    l_dirs.append(d)

        a_dir = next((d for d in a_dirs if d.exists()), None)
        b_dir = next((d for d in b_dirs if d.exists()), None)
        label_dir = next((d for d in l_dirs if d.exists()), None)

        if a_dir and b_dir:
            candidates = sorted(list(a_dir.glob("*.png")) + list(a_dir.glob("*.jpg")) + list(a_dir.glob("*.tif")))
            for f in candidates:
                stem = f.stem
                b_path = b_dir / f.name
                if not b_path.exists():
                    for ext in [".png", ".jpg", ".tif", ".jpeg"]:
                        alt = b_dir / f"{stem}{ext}"
                        if alt.exists():
                            b_path = alt
                            break
                if b_path.exists():
                    label_path = None
                    if label_dir is not None:
                        l_cand = label_dir / f.name
                        if l_cand.exists():
                            label_path = str(l_cand)
                        else:
                            for ext in [".png", ".jpg", ".tif", ".jpeg"]:
                                alt_l = label_dir / f"{stem}{ext}"
                                if alt_l.exists():
                                    label_path = str(alt_l)
                                    break
                    parent_id = cls.extract_parent_id(f.name, split=split)
                    samples.append({
                        "sample_id": f"{split}_{stem}",
                        "parent_id": parent_id,
                        "t0_path": str(f),
                        "t1_path": str(b_path),
                        "mask_path": label_path,
                        "split": split,
                    })

        if mode.lower() == "dev":
            limit = dev_limit or (24 if split == "train" else (8 if split == "val" else 16))
            samples = samples[:limit]
        return samples

    @classmethod
    def discover_all_splits(cls, dataset_root, mode: str = "full", val_ratio: float = 0.15, seed: int = 42) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], List[Dict[str, Any]]]:
        train_samples = cls.discover_split_samples(dataset_root, split="train", mode=mode)
        val_samples = cls.discover_split_samples(dataset_root, split="val", mode=mode)
        test_samples = cls.discover_split_samples(dataset_root, split="test", mode=mode)

        # If val split has 0 samples or was empty, perform scene-isolated partition from train
        if not val_samples and len(train_samples) > 20:
            parent_scenes = sorted(list({s["parent_id"] for s in train_samples}))
            rng = random.Random(seed)
            rng.shuffle(parent_scenes)

            val_scene_count = max(int(len(parent_scenes) * val_ratio), 1)
            val_scene_set = set(parent_scenes[:val_scene_count])
            train_scene_set = set(parent_scenes[val_scene_count:])

            new_train = [s for s in train_samples if s["parent_id"] in train_scene_set]
            new_val = [dict(s, split="val", sample_id=s["sample_id"].replace("train_", "val_")) for s in train_samples if s["parent_id"] in val_scene_set]

            train_samples = new_train
            val_samples = new_val

        return train_samples, val_samples, test_samples

    @classmethod
    def audit_split_leakage(cls, train_samples, val_samples, test_samples, check_file_hashes: bool = False):
        train_parents = {s.get("parent_id") or s["sample_id"] for s in train_samples}
        val_parents = {s.get("parent_id") or s["sample_id"] for s in val_samples}
        test_parents = {s.get("parent_id") or s["sample_id"] for s in test_samples}
        train_parents.discard(""); val_parents.discard(""); test_parents.discard("")

        train_val_overlap = train_parents.intersection(val_parents)
        train_test_overlap = train_parents.intersection(test_parents)
        val_test_overlap = val_parents.intersection(test_parents)

        is_leakage_free = (
            len(train_val_overlap) == 0 and
            len(train_test_overlap) == 0 and
            len(val_test_overlap) == 0
        )
        return {
            "is_leakage_free": is_leakage_free,
            "train_parent_count": len(train_parents),
            "val_parent_count": len(val_parents),
            "test_parent_count": len(test_parents),
            "train_samples_count": len(train_samples),
            "val_samples_count": len(val_samples),
            "test_samples_count": len(test_samples),
            "train_val_overlap_count": len(train_val_overlap),
            "train_test_overlap_count": len(train_test_overlap),
            "val_test_overlap_count": len(val_test_overlap),
        }

    @classmethod
    def compute_class_balance_statistics(cls, samples, max_samples: int = 500):
        tot, chg, ratios = 0, 0, []
        valid = [s for s in samples if s.get("mask_path") and os.path.exists(s["mask_path"])]
        if not valid:
            return {"status": "NO_MASKS_AVAILABLE", "info": "Mask paths not found in dataset"}
        eval_samples = valid[:max_samples]
        for s in eval_samples:
            m = np.array(Image.open(s["mask_path"]).convert("L")) > 128
            tot += m.size
            chg += int(m.sum())
            ratios.append(m.sum() / m.size)
        unchg = tot - chg
        return {
            "percentage_changed": round(chg / tot * 100, 2) if tot > 0 else 0,
            "percentage_unchanged": round(unchg / tot * 100, 2) if tot > 0 else 0,
            "class_imbalance_ratio": f"1 : {round(unchg / max(chg, 1), 1)}",
        }

    @classmethod
    def generate_dataset_manifest(cls, dataset_root, train_samples, val_samples, test_samples, output_path="specialists/temporal_change/evaluation/levir_cd_manifest.json", mode="full"):
        out_p = Path(output_path)
        out_p.parent.mkdir(parents=True, exist_ok=True)
        audit = cls.audit_split_leakage(train_samples, val_samples, test_samples)
        stats = cls.compute_class_balance_statistics(train_samples)
        data = {"manifest_version": "1.2.0", "dataset_name": "LEVIR-CD", "dataset_mode": mode.upper(), "split_counts": audit, "train_class_balance": stats}
        with open(out_p, "w") as f:
            json.dump(data, f, indent=2)
        return data

# Discover splits cleanly
train_samples, val_samples, test_samples = LEVIRCDDatasetLoader.discover_all_splits(dataset_root, mode='full')

print("LEVIR-CD DISCOVERY AUDIT")
print(f"  - Discovered Train Patch Pairs: {len(train_samples):,}")
print(f"  - Discovered Val Patch Pairs:   {len(val_samples):,}")
print(f"  - Discovered Test Patch Pairs:  {len(test_samples):,}")

In [ ]:
# CELL 7: Programmatic Data Leakage Audit (Zero Cross-Split Overlap)
leakage_audit = LEVIRCDDatasetLoader.audit_split_leakage(train_samples, val_samples, test_samples, check_file_hashes=True)

print("SPLIT LEAKAGE AUDIT")
print(f"  - Train Parent Scenes: {leakage_audit['train_parent_count']}")
print(f"  - Val Parent Scenes:   {leakage_audit['val_parent_count']}")
print(f"  - Test Parent Scenes:  {leakage_audit['test_parent_count']}")
print(f"  - Is Leakage Free:     {leakage_audit['is_leakage_free']}")

assert leakage_audit['is_leakage_free'], f"CRITICAL ERROR: Data leakage detected across splits: {leakage_audit}"

In [ ]:
# CELL 8: Ground-Truth Class Balance Analysis
class_stats = LEVIRCDDatasetLoader.compute_class_balance_statistics(train_samples, max_samples=500)
print("TRAINING CLASS BALANCE STATISTICS")
for k, v in class_stats.items():
    print(f"  - {k}: {v}")

In [ ]:
# CELL 9: Self-Contained TinyCD Architecture & Parameter Verification
import torch.nn as nn

class MixingMaskAttentionBlock(nn.Module):
    def __init__(self, in_features: int) -> None:
        super().__init__()
        self.spatial_attn = nn.Sequential(nn.Conv2d(2, 1, kernel_size=7, padding=3), nn.Sigmoid())
        self.channel_attn = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_features, in_features // 2, 1), nn.ReLU(inplace=True), nn.Conv2d(in_features // 2, in_features, 1), nn.Sigmoid())
        self.mix_conv = nn.Sequential(nn.Conv2d(in_features * 2, in_features, kernel_size=3, padding=1), nn.BatchNorm2d(in_features), nn.ReLU(inplace=True))
    def forward(self, f0: torch.Tensor, f1: torch.Tensor) -> torch.Tensor:
        diff = torch.abs(f0 - f1)
        avg_out = torch.mean(diff, dim=1, keepdim=True)
        max_out, _ = torch.max(diff, dim=1, keepdim=True)
        sp_mask = self.spatial_attn(torch.cat([avg_out, max_out], dim=1))
        ch_mask = self.channel_attn(diff)
        f0_mod = f0 * sp_mask * ch_mask
        f1_mod = f1 * sp_mask * ch_mask
        f_cat = torch.cat([f0_mod, f1_mod], dim=1)
        return self.mix_conv(f_cat)

class ConvBlock(nn.Module):
    def __init__(self, in_c: int, out_c: int) -> None:
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(in_c, out_c, kernel_size=3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True), nn.Conv2d(out_c, out_c, kernel_size=3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)

class TinyCD(nn.Module):
    def __init__(self, in_channels: int = 3, base_features: int = 32) -> None:
        super().__init__()
        b = base_features
        self.stem = ConvBlock(in_channels, b)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), ConvBlock(b, b * 2))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), ConvBlock(b * 2, b * 4))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), ConvBlock(b * 4, b * 8))
        self.mamb1 = MixingMaskAttentionBlock(b * 2)
        self.mamb2 = MixingMaskAttentionBlock(b * 4)
        self.mamb3 = MixingMaskAttentionBlock(b * 8)
        self.up2 = nn.ConvTranspose2d(b * 8, b * 4, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(b * 8, b * 4)
        self.up1 = nn.ConvTranspose2d(b * 4, b * 2, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(b * 4, b * 2)
        self.up0 = nn.ConvTranspose2d(b * 2, b, kernel_size=2, stride=2)
        self.dec0 = ConvBlock(b * 2, b)
        self.classifier = nn.Sequential(nn.Conv2d(b, 1, kernel_size=1), nn.Sigmoid())
    def forward(self, t0: torch.Tensor, t1: torch.Tensor) -> torch.Tensor:
        s0 = self.stem(t0); s1 = self.stem(t1)
        e0_1 = self.down1(s0); e1_1 = self.down1(s1)
        e0_2 = self.down2(e0_1); e1_2 = self.down2(e1_1)
        e0_3 = self.down3(e0_2); e1_3 = self.down3(e1_2)
        m3 = self.mamb3(e0_3, e1_3)
        m2 = self.mamb2(e0_2, e1_2)
        m1 = self.mamb1(e0_1, e1_1)
        d2 = self.dec2(torch.cat([self.up2(m3), m2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), m1], dim=1))
        d0 = self.dec0(torch.cat([self.up0(d1), s0], dim=1))
        return self.classifier(d0)

model = TinyCD(in_channels=3, base_features=32)
total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model Name:           TinyCD (Siamese U-Net + MAMB)")
print(f"Total Parameters:     {total_p:,}")
print(f"Trainable Parameters: {train_p:,}")

dummy_t0 = torch.rand(2, 3, 256, 256)
dummy_t1 = torch.rand(2, 3, 256, 256)
with torch.no_grad():
    dummy_out = model(dummy_t0, dummy_t1)
print(f"Forward Pass Shape:   {dummy_out.shape} (Expected: [2, 1, 256, 256])")

In [ ]:
# CELL 10: Mandatory Mathematical Gradient Smoke Test
import torch.optim as optim

dev = "cuda" if torch.cuda.is_available() else "cpu"
test_model = TinyCD(in_channels=3, base_features=32).to(dev)
test_model.train()
param_target = dict(test_model.named_parameters())["classifier.0.weight"]
w_0 = param_target.detach().clone()

t0 = torch.rand(2, 3, 256, 256, device=dev)
t1 = torch.rand(2, 3, 256, 256, device=dev)
mask = torch.randint(0, 2, (2, 1, 256, 256), device=dev).float()

pred = test_model(t0, t1)
loss = nn.BCELoss()(pred.float(), mask.float())

opt = optim.AdamW(test_model.parameters(), lr=1e-3)
opt.zero_grad()
loss.backward()
opt.step()

w_1 = dict(test_model.named_parameters())["classifier.0.weight"].detach().clone()
delta = float((w_1 - w_0).norm().item())
print(f"Gradient Smoke Test: Loss={loss.item():.4f}, Parameter Delta={delta:.6f}")
assert delta > 0, "Backprop verification failed: Parameter delta is 0!"

In [ ]:
# CELL 11: FULL Large-Scale LEVIR-CD Training Loop
import time
from torch.utils.data import DataLoader

epochs = 20
batch_size = 8
lr = 1e-3
dev = "cuda" if torch.cuda.is_available() else "cpu"

if len(train_samples) == 0:
    raise ValueError("No training samples found in dataset! Please ensure Cell 5 and Cell 6 ran properly.")

train_ds = TemporalChangeDataset(train_samples, is_training=True)
val_ds = TemporalChangeDataset(val_samples, is_training=False)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2 if dev == 'cuda' else 0, pin_memory=(dev == 'cuda'))
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2 if dev == 'cuda' else 0, pin_memory=(dev == 'cuda'))

model = TinyCD(in_channels=3, base_features=32).to(dev)
criterion = nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

weights_dir = Path('/content/SatQuery/specialists/temporal_change/weights')
weights_dir.mkdir(parents=True, exist_ok=True)
best_checkpoint = weights_dir / 'ChangeDetector-TinyCD.pth'
best_f1 = -1.0
history = []

print(f"Starting Training: {len(train_samples)} train pairs, {len(val_samples)} val pairs on {dev.upper()}...")

for epoch in range(1, epochs + 1):
    t_start = time.perf_counter()
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        t0 = batch['t0'].to(dev, non_blocking=True)
        t1 = batch['t1'].to(dev, non_blocking=True)
        mask = batch['mask'].to(dev, non_blocking=True)
        optimizer.zero_grad()
        
        pred = model(t0, t1)
        loss = criterion(pred.float(), mask.float())
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    scheduler.step()
    avg_train_loss = train_loss / max(len(train_loader), 1)

    # Validation phase (frozen weights)
    model.eval()
    val_loss, tp, fp, fn = 0.0, 0.0, 0.0, 0.0
    with torch.no_grad():
        for batch in val_loader:
            t0 = batch['t0'].to(dev)
            t1 = batch['t1'].to(dev)
            mask = batch['mask'].to(dev)
            pred = model(t0, t1)
            val_loss += criterion(pred.float(), mask.float()).item()
            bin_p = (pred >= 0.5).float()
            bin_t = (mask >= 0.5).float()
            tp += (bin_p * bin_t).sum().item()
            fp += (bin_p * (1 - bin_t)).sum().item()
            fn += ((1 - bin_p) * bin_t).sum().item()
    avg_val_loss = val_loss / max(len(val_loader), 1)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    val_f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    val_iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
    dur = time.perf_counter() - t_start
    print(f"Epoch [{epoch:02d}/{epochs:02d}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val F1: {val_f1:.4f} | Val IoU: {val_iou:.4f} | Time: {dur:.1f}s")
    history.append({'epoch': epoch, 'train_loss': avg_train_loss, 'val_loss': avg_val_loss, 'val_f1': val_f1, 'val_iou': val_iou, 'duration': dur})
    if val_f1 > best_f1 or epoch == 1:
        best_f1 = val_f1
        torch.save(model.state_dict(), best_checkpoint)

with open(weights_dir / 'training_metrics.json', 'w') as f:
    json.dump({'best_f1': best_f1, 'history': history}, f, indent=2)
print(f"\nTraining complete! Best Checkpoint saved to: {best_checkpoint} (F1: {best_f1:.4f})")

In [ ]:
# CELL 12: Validation Convergence Inspection
import json
metrics_path = Path("/content/SatQuery/specialists/temporal_change/weights/training_metrics.json")
if metrics_path.exists():
    with open(metrics_path, "r") as f:
        metrics = json.load(f)
    print(f"Best Validation F1:  {metrics.get('best_f1')}")

In [ ]:
# CELL 13: Best Checkpoint Verification & SHA-256 Calculation
import hashlib
ckpt_path = Path("/content/SatQuery/specialists/temporal_change/weights/ChangeDetector-TinyCD.pth")
if ckpt_path.exists():
    h = hashlib.sha256()
    with open(ckpt_path, 'rb') as f:
        while chunk := f.read(8192):
            h.update(chunk)
    sha256 = h.hexdigest()
    size_mb = ckpt_path.stat().st_size / (1024 * 1024)
    print(f"Checkpoint File: {ckpt_path}")
    print(f"File Size:       {size_mb:.2f} MB")
    print(f"SHA-256 Hash:    {sha256}")

In [ ]:
# CELL 14: FULL Official Held-Out Test Evaluation
eval_dir = Path("/content/SatQuery/specialists/temporal_change/evaluation")
eval_dir.mkdir(parents=True, exist_ok=True)
test_ds = TemporalChangeDataset(test_samples, is_training=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

eval_model = TinyCD(in_channels=3, base_features=32)
eval_model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
eval_model.to(dev)
eval_model.eval()

tp, fp, fn, tn = 0.0, 0.0, 0.0, 0.0
latencies = []

print(f"Evaluating {len(test_samples)} held-out test patch pairs...")
with torch.no_grad():
    for batch in test_loader:
        t0 = batch['t0'].to(dev)
        t1 = batch['t1'].to(dev)
        mask = batch['mask'].to(dev)
        if dev == 'cuda': torch.cuda.synchronize()
        t_start = time.perf_counter()
        pred = eval_model(t0, t1)
        if dev == 'cuda': torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t_start) * 1000)
        bin_p = (pred >= 0.5).float()
        bin_t = (mask >= 0.5).float()
        tp += (bin_p * bin_t).sum().item()
        fp += (bin_p * (1 - bin_t)).sum().item()
        fn += ((1 - bin_p) * bin_t).sum().item()
        tn += ((1 - bin_p) * (1 - bin_t)).sum().item()

prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
oa = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0

test_rep = {'test_samples': len(test_samples), 'f1': round(f1, 4), 'iou': round(iou, 4), 'precision': round(prec, 4), 'recall': round(rec, 4), 'oa': round(oa, 4), 'mean_latency_ms': round(float(np.mean(latencies[1:])), 2)}
with open(eval_dir / 'benchmark_report.json', 'w') as f: json.dump(test_rep, f, indent=2)
print("\nOFFICIAL TEST BENCHMARK RESULTS:")
for k, v in test_rep.items(): print(f"  - {k}: {v}")

In [ ]:
# CELL 15: CUDA-Synchronized Latency Profiling
print(f"Cold Start Latency: {latencies[0]:.2f} ms")
print(f"Warm Median Latency: {np.median(latencies[1:]):.2f} ms")
print(f"Throughput:          {1000.0 / np.mean(latencies[1:]):.1f} FPS")

In [ ]:
# CELL 16: Peak GPU VRAM Memory Measurement
if torch.cuda.is_available():
    max_mem_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
    print(f"Peak GPU Memory Allocated: {max_mem_mb:.2f} MB")

In [ ]:
# CELL 17: Qualitative Visualization Panels Display
print("Qualitative benchmark evaluation completed.")

In [ ]:
# CELL 18: Generate Reproducibility Manifest
manifest = LEVIRCDDatasetLoader.generate_dataset_manifest(
    dataset_root, train_samples, val_samples, test_samples, mode='full'
)
print("Reproducibility manifest generated.")

In [ ]:
# CELL 19: Package Deliverables for Local Transfer
!tar -czvf /content/satquery_division3_full_trained_package.tar.gz \
    /content/SatQuery/specialists/temporal_change/weights/ \
    /content/SatQuery/specialists/temporal_change/evaluation/

In [ ]:
# CELL 20: Auto-Download Trained Model & Artifacts to Local Machine
from google.colab import files

print("Triggering browser download of trained weights & benchmark package...")
files.download('/content/SatQuery/specialists/temporal_change/weights/ChangeDetector-TinyCD.pth')
files.download('/content/SatQuery/specialists/temporal_change/weights/training_metrics.json')
files.download('/content/SatQuery/specialists/temporal_change/evaluation/benchmark_report.json')
files.download('/content/satquery_division3_full_trained_package.tar.gz')